# Clase 5 · E07: lectura, transformación y ejecución con Spark
Viernes 16 de octubre de 2026: demostración docente; sábado 17: ejecución propia y entrega en clase. Preparado por el PhD Esteban Hernández, CyberColombia.

[Taller E07](../../talleres/E07.md) · [WSL](../../docs/instalacion-wsl.md) · [Linux/macOS](../../docs/instalacion-linux-macos.md)

**Antes del receso:** datos, tipos, Parquet, sesión Spark, transformaciones y acciones.
**Después del receso:** agregación, plan, materialización, equivalencia DuckDB y evidencias.

## Preparación
Usar el `.venv` de la raíz y JDK 21 según la guía del sistema. Desde la raíz del repositorio, con el entorno activo:
```bash
python -m pip install -r kit/requirements_spark.txt
java -version
cd kit
python 00_datos.py --descargar eva divipola
python 00_datos.py --verificar eva divipola
python -m jupyter lab
```
Seleccionar el kernel del mismo entorno. Si `JAVA_HOME` no está configurado, aplicar la guía antes de abrir Jupyter; reiniciar el servidor para heredar las variables. Trabajar en `local[2]`, incluso con 8 GB de RAM.

Este notebook prepara sus propios productos desde los CSV originales, sin ejecutar `talleres.py`. Las salidas están en `kit/salidas/E07`; las escrituras reemplazan únicamente los productos homónimos de este ejercicio. Los originales no se modifican.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
import duckdb
from IPython.display import display
repositorio = Path("/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate")
# Linux/macOS: repositorio = Path.home() / "bigdata/big-data-postgraduate"
KIT = repositorio / "kit"
RAW = KIT / "data/raw"
OUT = KIT / "salidas/E07"
assert KIT.is_dir(), "Ajusta repositorio a tu clon local"
OUT.mkdir(parents=True, exist_ok=True)
# Los workers usan el mismo Python del kernel y su .venv.
os.environ["PYSPARK_PYTHON"] = sys.executable
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("E07-aprendizaje")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC").getOrCreate())
print("Python:", sys.executable, "Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Shuffle:", spark.conf.get("spark.sql.shuffle.partitions"))

## 1. Leer como texto e inspeccionar
Conservar códigos con ceros iniciales. Primero ver columnas, valores y tipos; luego elegir los campos. La preparación pandas es local y se mide separada del procesamiento Spark.

In [ ]:
eva_texto = pd.read_csv(RAW / "eva.csv", dtype="string", keep_default_na=False)
divipola = pd.read_csv(RAW / "divipola.csv", dtype="string", keep_default_na=False)
display(eva_texto.head())
display(eva_texto.dtypes)
display(divipola.head())
print("Filas originales:", len(eva_texto), "DIVIPOLA:", len(divipola))

## 2. Convertir y conservar los problemas
Renombrar, convertir con `to_numeric` y marcar el dominio del cociente: área positiva, producción no negativa y año interpretable. No calcular un promedio simple de rendimientos. Las filas fuera del dominio se guardan para revisión con su número de fila; no se borran del producto EVA.

In [ ]:
renombres = {"Código Dane municipio": "codigo_municipio", "Código Dane departamento": "codigo_departamento",
    "Año": "anio", "Cultivo": "cultivo", "Periodo": "periodo", "Estado físico del cultivo": "estado_fisico",
    "Área cosechada": "area_ha", "Producción": "produccion_t", "Rendimiento": "rendimiento_publicado"}
eva = eva_texto[list(renombres)].rename(columns=renombres).copy()
eva["fila_fuente"] = range(1, len(eva) + 1)
for campo, ancho in [("codigo_municipio", 5), ("codigo_departamento", 2)]:
    eva[campo] = eva[campo].str.strip().str.zfill(ancho)
for campo in ["area_ha", "produccion_t", "rendimiento_publicado", "anio"]:
    eva[campo + "_texto"] = eva[campo]
    eva[campo] = pd.to_numeric(eva[campo], errors="coerce")
    errores = eva[campo].isna() & eva[campo + "_texto"].str.strip().ne("")
    print(campo, "no convertibles:", int(errores.sum()))
    display(eva.loc[errores, ["fila_fuente", campo + "_texto"]].head())
# Tipos estables en Parquet, compatibles con Spark y DuckDB.
assert eva["anio"].dropna().mod(1).eq(0).all(), "Revisar años fraccionarios"
eva["anio"] = eva["anio"].astype("Int64")
for campo in ["area_ha", "produccion_t", "rendimiento_publicado"]:
    eva[campo] = eva[campo].astype("Float64")
eva["apto_rendimiento"] = (eva["area_ha"].gt(0) & eva["produccion_t"].ge(0) & eva["anio"].notna()).fillna(False)
revision = eva.loc[~eva["apto_rendimiento"]].copy()
revision.to_csv(OUT / "eva_revision.csv", index=False)
assert len(eva) == int(eva["apto_rendimiento"].sum()) + len(revision)
assert int(eva["apto_rendimiento"].sum()) == 159616, "Control del corte docente; revisar fuente si cambió"
display(revision.head())

## 3. Escribir y volver a leer los Parquet
Preparar una dimensión única por código para el siguiente notebook. No unir todavía EVA con la dimensión; primero comprobar su cardinalidad.

In [ ]:
dimension = divipola.rename(columns={"Código Municipio": "codigo_municipio",
    "Código Departamento": "codigo_departamento", "Nombre Municipio": "municipio",
    "Nombre Departamento": "departamento"})[
        ["codigo_municipio", "codigo_departamento", "municipio", "departamento"]].copy()
dimension["codigo_municipio"] = dimension["codigo_municipio"].str.zfill(5)
dimension["codigo_departamento"] = dimension["codigo_departamento"].str.zfill(2)
assert dimension["codigo_municipio"].notna().all()
assert dimension["codigo_municipio"].str.fullmatch(r"\d{5}").all()
assert dimension["codigo_municipio"].is_unique
archivo_eva = OUT / "eva.parquet"
eva.to_parquet(archivo_eva, index=False)
dimension.to_parquet(OUT / "dim_municipio.parquet", index=False)
pd.testing.assert_frame_equal(eva, pd.read_parquet(archivo_eva))
pd.testing.assert_frame_equal(dimension, pd.read_parquet(OUT / "dim_municipio.parquet"))
print("Parquet verificado:", archivo_eva)

## 4. Leer con Spark: transformaciones y acciones
`read.parquet` obtiene el esquema; `filter` y `select` construyen un plan. `show`, `count`, escritura y `collect` disparan trabajo. No usar `toPandas()` sobre todos los datos sin evaluar memoria. Aquí solo se visualizan cinco filas.

In [ ]:
eva_spark = spark.read.parquet(str(archivo_eva))
eva_spark.printSchema()
aptos = eva_spark.filter(F.col("apto_rendimiento"))
aptos.select("codigo_municipio", "cultivo", "anio", "area_ha", "produccion_t").show(5, truncate=False)
print("Particiones de entrada:", eva_spark.rdd.getNumPartitions())
assert aptos.count() == 159616

## 5. Agregar sin ejecutar todavía
La unidad es municipio–cultivo–estado físico–año. Calcular suma de producción / suma de área. En `explain` localizar lectura, filtro, agregaciones y `Exchange` (redistribución o shuffle). Explicar por qué cuatro particiones de shuffle no equivalen a cuatro máquinas.

In [ ]:
claves = ["codigo_municipio", "cultivo", "estado_fisico", "anio"]
rendimiento = (aptos.groupBy(*claves)
    .agg(F.sum("produccion_t").alias("produccion_t"), F.sum("area_ha").alias("area_ha"))
    .withColumn("rendimiento_t_ha", F.col("produccion_t") / F.col("area_ha")))
rendimiento.explain("formatted")

## 6. Medir una acción completa
La siguiente celda incluye lectura del Parquet, filtro, agregación y escritura del resultado. La sesión Spark ya está iniciada. No mide solo la construcción del plan ni incluye su arranque. Las celdas de verificación y visualización se ejecutan aparte.

In [ ]:
%%time
rendimiento.write.mode("overwrite").parquet(str(OUT / "spark_rendimiento"))

In [ ]:
rendimiento_guardado = spark.read.parquet(str(OUT / "spark_rendimiento"))
assert rendimiento_guardado.count() == 107620
rendimiento_guardado.orderBy(*claves).show(5, truncate=False)
archivos = sorted((OUT / "spark_rendimiento").glob("part-*.parquet"))
print("Archivos de datos:", len(archivos))
# La carpeta completa es el dataset; no elegir solo el primer part-*.

## 7. Consulta equivalente con DuckDB
Usar el mismo Parquet, predicado y claves. El tiempo incluye lectura, agregación y escritura; no la preparación pandas. DuckDB escribe un archivo y Spark un dataset: reportar esta diferencia y no sacar conclusiones de escalabilidad de una ejecución local.

In [ ]:
con = duckdb.connect()
con.execute("SET threads=2")
con.read_parquet(str(archivo_eva)).create_view("eva")
sql = 'SELECT codigo_municipio, cultivo, estado_fisico, anio,\n       SUM(produccion_t) AS produccion_t, SUM(area_ha) AS area_ha,\n       SUM(produccion_t) / SUM(area_ha) AS rendimiento_t_ha\nFROM eva WHERE apto_rendimiento\nGROUP BY codigo_municipio, cultivo, estado_fisico, anio'

In [ ]:
%%time
referencia = con.sql(sql).df()
referencia.to_parquet(OUT / "rendimiento_duckdb.parquet", index=False)

## 8. Comparar cada grupo, no solo el total
Este resultado agregado cabe en memoria local. Comprobar claves únicas, cobertura idéntica y las tres medidas numéricas con tolerancias explícitas. Ordenar elimina diferencias de orden entre motores; no redondear antes de comparar.

In [ ]:
resultado = pd.read_parquet(OUT / "spark_rendimiento")
assert not referencia.duplicated(claves).any() and not resultado.duplicated(claves).any()
cobertura = referencia[claves].merge(resultado[claves], on=claves, how="outer", validate="one_to_one", indicator=True)
assert cobertura["_merge"].eq("both").all()
columnas = claves + ["produccion_t", "area_ha", "rendimiento_t_ha"]
a = referencia[columnas].sort_values(claves).reset_index(drop=True)
b = resultado[columnas].sort_values(claves).reset_index(drop=True)
pd.testing.assert_frame_equal(a[claves], b[claves], check_dtype=False)
import numpy as np
diferencias = {}
for campo in ["produccion_t", "area_ha", "rendimiento_t_ha"]:
    distintos = ~np.isclose(a[campo], b[campo], atol=0.000001, rtol=0, equal_nan=False)
    diferencias[campo] = int(distintos.sum())
assert not any(diferencias.values()), diferencias
control = {"filas_eva": len(eva), "aptos_rendimiento": int(eva["apto_rendimiento"].sum()),
    "grupos": len(resultado), "diferencias_duckdb": diferencias, "tolerancia_absoluta": 0.000001,
    "spark": spark.version, "duckdb": duckdb.__version__, "master": spark.sparkContext.master,
    "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions")}
with open(OUT / "control_spark.json", "w", encoding="utf-8") as archivo:
    json.dump(control, archivo, ensure_ascii=False, indent=2)
display(control)
con.close()

## 9. Evidencia y continuación
Guardar el notebook ejecutado con el plan y explicar dónde se lee, filtra, redistribuye y agrega. Entregar el control de equivalencia y dos observaciones sobre el coste de ejecución local. Los controles numéricos corresponden al corte docente, no a todas las versiones futuras de EVA.

Continuar con [06 Joins y ventanas](06_Spark_joins_y_ventanas.ipynb). Detener la sesión antes de abrirlo; su primera celda crea una sesión nueva. Para repetir este notebook, iniciar desde la configuración.

In [ ]:
spark.stop()